# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "CID": "customer_key",
    "CNTRY": "country"
}

# Reading from Bronze layer

In [0]:
df = spark.table("data_lakehouse_project.bronze.erp_loc_a101")

# Data Transformation

- TRIM the strings
- Upper Column CID
- Column CNTRY (Nulls,)
- Rename Columns

In [0]:
df.display()

## Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Upper CID column + Standardized to match other table

In [0]:
df = df.withColumn("CID", F.upper(col("CID")))

In [0]:
df = df.withColumn("CID", F.regexp_replace(F.col("CID"), "-|NAS", ""))

## CNTRY

In [0]:
df = df.withColumn(
    "CNTRY",
    F.when(F.col("CNTRY").isNull(), "N/A")
     .when(F.trim(F.col("CNTRY")) == "", "N/A")
     .when(F.col("CNTRY").rlike("(?i)^(US|United States)"), "USA")
     .when(F.col("CNTRY").rlike("(?i)^(DE|Germany)"), "Germany")
     .otherwise(F.col("CNTRY"))
)

## Rename Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("data_lakehouse_project.silver.erp_loc_a101")
)

# Check queries

In [0]:
%sql
SELECT *
FROM data_lakehouse_project.silver.erp_loc_a101

